# Hito 2 — Midpoint Model + Error Analysis

**IIT414W — Capstone: F1 Race Strategy Advisor**
*Week 11 — Hito 2 due Wed May 13, 23:59 CLT*

---

## What this notebook is

A reproducible Hito 2 notebook that trains two calibrated targets, compares them side by side, and slices error analysis by strategy type, circuit type, and constructor tier. The fixed expansion target for this deliverable is **is_top3**.

## What this notebook is NOT

- Not a causal claim about strategy. Strategy inputs are treated as what-if scenario controls, not proof of causal effect.
- Not a single-target notebook. Hito 2 requires both **is_top10** and **is_top3** to be modeled and compared.

## How to use it

1. Run **Step 1** to load the race-level dataset with the locked split.
2. Run **Step 2** to verify the leakage guard.
3. Run **Step 3** to train both calibrated models and report test metrics.
4. Run **Step 4** and **Step 5** to score the matched what-if pair.
5. Run **Step 6** to produce the error analysis tables used in the markdown deliverables.

## The one rule that matters

Use scenario-conditioned language, not causal language. The notebook compares scenarios under a fixed driver-race context and reports how the two targets respond. If the targets disagree, that disagreement is the finding.


In [2]:
# ===== Imports and configuration =====
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
    mean_absolute_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# scikit-learn >= 1.6 has FrozenEstimator. If your environment is older, see
# the fallback comment in Step 3.
try:
    from sklearn.frozen import FrozenEstimator
    _HAS_FROZEN = True
except ImportError:
    _HAS_FROZEN = False

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

# ===== COURSE CONSTANTS — DO NOT CHANGE =====
RANDOM_SEED = 414
TRAIN_SEASONS = [2019, 2020, 2021]
CALIB_SEASONS = [2022]
TEST_SEASONS = [2023, 2024]
PRIMARY_TARGET = "is_top10"
EXPANSION_TARGET = "is_top3"
DOCENT_BRIER = 0.132
DOCENT_ROC_AUC = 0.892

# ===== TEAM CONFIG =====
DATA_PATH = "../f1_strategy_race_level.csv"

assert EXPANSION_TARGET == "is_top3", "Hito 2 requires is_top3 as the expansion target."

print(f"Primary target:    {PRIMARY_TARGET} (binary)")
print(f"Expansion target:  {EXPANSION_TARGET} (binary)")
print(f"Random seed:       {RANDOM_SEED}")
print(f"Docent reference:  Brier={DOCENT_BRIER}  ROC-AUC={DOCENT_ROC_AUC}  (is_top10 only)")


Primary target:    is_top10 (binary)
Expansion target:  is_top3 (binary)
Random seed:       414
Docent reference:  Brier=0.132  ROC-AUC=0.892  (is_top10 only)


## Step 1 — Load the dataset and apply the locked temporal split

The split is locked: **train = 2019–2021 · calibration = 2022 · test = 2023–2024**. The expansion target is fixed to **is_top3**.


In [3]:
# ===== Step 1 — Load + locked split =====
df = pd.read_csv('f1_strategy_race_level.csv')

train = df[df["season"].isin(TRAIN_SEASONS)].copy()
calib = df[df["season"].isin(CALIB_SEASONS)].copy()
test  = df[df["season"].isin(TEST_SEASONS)].copy()

print()
print(f"Train (2019-2021):  {len(train):>5,} rows")
print(f"Calib (2022):       {len(calib):>5,} rows")
print(f"Test  (2023-2024):  {len(test):>5,} rows")

print()
print("Target check on test set (2023-2024):")
print(f"  is_top10 rate:  {test['is_top10'].mean():.3f}")
print(f"  is_top3 rate:   {test['is_top3'].mean():.3f}")



Train (2019-2021):  1,132 rows
Calib (2022):         426 rows
Test  (2023-2024):    889 rows

Target check on test set (2023-2024):
  is_top10 rate:  0.517
  is_top3 rate:   0.155


## Step 2 — Leakage guard

The model is allowed to use pre-race signals and scenario inputs, but not audit columns or outcomes. Strategy variables remain scenario inputs in the what-if analysis, not causal proof.


In [ ]:
# ===== Step 2 — Leakage guard cell =====
print("=== LEAKAGE GUARD — column classification ===")
print("Strategy features are treated as scenario inputs for what-if scoring only. They are not claimed as causal drivers.")

COLUMN_CLASS = {
    "season": "id",
    "round": "id",
    "circuit": "id",
    "circuit_type": "pre_race",
    "Driver": "id",
    "Team": "id",
    "grid_position": "pre_race",
    "qualifying_position": "pre_race",
    "qualifying_time_s": "pre_race",
    "constructor_tier": "pre_race",
    "driver_prior3_avg_finish": "pre_race",
    "constructor_prior3_avg_finish": "pre_race",
    "driver_circuit_prior_avg": "pre_race",
    "n_stops": "scenario_input",
    "strategy_type": "scenario_input",
    "compound_sequence": "scenario_input",
    "stint1_length": "scenario_input",
    "stint2_length": "scenario_input",
    "stint3_length": "scenario_input",
    "stint4_length": "scenario_input",
    "stint5_length": "scenario_input",
    "avg_pit_stop_duration_s": "scenario_input",
    "total_pit_time_s": "scenario_input",
    "first_pit_lap": "scenario_input",
    "last_pit_lap": "scenario_input",
    "weather_actual": "audit",
    "safety_car_periods": "audit",
    "safety_car_laps": "audit",
    "vsc_laps": "audit",
    "wet_laps": "audit",
    "avg_track_temp": "audit",
    "avg_air_temp": "audit",
    "is_top10": "outcome",
    "is_top3": "outcome",
    "finish_position": "outcome",
    "points": "outcome",
    "positions_gained": "outcome",
    "dnf": "outcome",
    "status": "outcome",
}

classified = set(COLUMN_CLASS) & set(df.columns)
unclassified = sorted(set(df.columns) - set(COLUMN_CLASS))
by_class = {"pre_race": [], "scenario_input": [], "audit": [], "outcome": [], "id": []}
for c in classified:
    by_class[COLUMN_CLASS[c]].append(c)

for cls in ["pre_race", "scenario_input", "audit", "outcome", "id"]:
    cols = sorted(by_class[cls])
    print(f"\n[{cls}] ({len(cols)})")
    for c in cols:
        print(f"  - {c}")

if unclassified:
    print(f"\n[unclassified] ({len(unclassified)}) — review before fitting")
    for c in unclassified:
        print(f"  - {c}")
else:
    print("\nAll columns in the dataset are classified.")


## Step 3 — Train two models on the same features and the same split

We train one calibrated model for **is_top10** and one calibrated model for **is_top3**. The feature set is identical for both targets, so any disagreement comes from the target definition rather than from a different predictor set.


In [ ]:
# ===== Step 3 — Build pipeline and train both models =====
# Feature set includes pre-race covariates and strategy inputs so the model can score counterfactual what-if rows.

NUMERIC_FEATURES = [
    "grid_position",
    "driver_prior3_avg_finish",
    "constructor_prior3_avg_finish",
    "driver_circuit_prior_avg",
    "n_stops",
    "stint1_length",
    "stint2_length",
    "stint3_length",
    "avg_pit_stop_duration_s",
    "total_pit_time_s",
    "first_pit_lap",
    "last_pit_lap",
]
CATEGORICAL_FEATURES = [
    "circuit_type",
    "constructor_tier",
    "strategy_type",
    "compound_sequence",
]
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), NUMERIC_FEATURES),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_ohe()),
            ]), CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )

def fit_binary(target_name: str):
    base = Pipeline([
        ("prep", build_preprocessor()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
    ])
    base.fit(train[FEATURE_COLS], train[target_name])
    if _HAS_FROZEN:
        calibrated = CalibratedClassifierCV(estimator=FrozenEstimator(base), method="sigmoid")
        calibrated.fit(calib[FEATURE_COLS], calib[target_name])
        return calibrated
    calibrated = CalibratedClassifierCV(estimator=base, method="sigmoid", cv="prefit")
    calibrated.fit(calib[FEATURE_COLS], calib[target_name])
    return calibrated

def evaluate_binary(model, target_name: str) -> dict:
    proba = model.predict_proba(test[FEATURE_COLS])[:, 1]
    y = test[target_name]
    return {
        "target": target_name,
        "kind": "binary",
        "brier": float(brier_score_loss(y, proba)),
        "log_loss": float(log_loss(y, proba)),
        "roc_auc": float(roc_auc_score(y, proba)),
        "test_positive_rate": float(y.mean()),
    }

print(f"Fitting primary model on {PRIMARY_TARGET}...")
model_primary = fit_binary(PRIMARY_TARGET)
metrics_primary = evaluate_binary(model_primary, PRIMARY_TARGET)

print(f"Fitting expansion model on {EXPANSION_TARGET}...")
model_expansion = fit_binary(EXPANSION_TARGET)
metrics_expansion = evaluate_binary(model_expansion, EXPANSION_TARGET)

print("\n=== Metrics on test set (2023-2024) ===")
print(f"Primary target: {PRIMARY_TARGET}")
print(f"  Brier    = {metrics_primary['brier']:.4f}")
print(f"  Log loss = {metrics_primary['log_loss']:.4f}")
print(f"  ROC-AUC  = {metrics_primary['roc_auc']:.4f}")
print(f"  Pos. rate= {metrics_primary['test_positive_rate']:.3f}")
print(f"\nExpansion target: {EXPANSION_TARGET}")
print(f"  Brier    = {metrics_expansion['brier']:.4f}")
print(f"  Log loss = {metrics_expansion['log_loss']:.4f}")
print(f"  ROC-AUC  = {metrics_expansion['roc_auc']:.4f}")
print(f"  Pos. rate= {metrics_expansion['test_positive_rate']:.3f}")


## Step 4 — Define the matched what-if scenario pair

This is the step that surfaces the recommendation the top-10 target alone would miss. The default pair is chosen so **is_top10** and **is_top3** disagree on the better strategy.


In [ ]:
# ===== Step 4 — score_pair() and three example pairs =====

def score_pair(
    df_test: pd.DataFrame,
    context_filter: dict,
    scenario_a: dict,
    scenario_b: dict,
    *,
    label_a: str = "Scenario A",
    label_b: str = "Scenario B",
) -> pd.DataFrame:
    mask = pd.Series(True, index=df_test.index)
    for col, val in context_filter.items():
        if isinstance(val, str) and col in df_test.columns and df_test[col].dtype == object:
            mask &= df_test[col].astype(str).str.contains(val, case=False, na=False)
        else:
            mask &= (df_test[col] == val)
    matching = df_test[mask]
    if matching.empty:
        raise ValueError(f"No row in test set matches context_filter={context_filter}.")
    base_row = matching.iloc[0]

    for label, scn in [(label_a, scenario_a), (label_b, scenario_b)]:
        for k in scn:
            cls = COLUMN_CLASS.get(k, "unclassified")
            if cls != "scenario_input":
                raise ValueError(
                    f"In {label}: {k!r} is class {cls!r}, not scenario_input. "
                    "What-if comparisons must vary only scenario inputs."
                )

    row_a = base_row.copy()
    for k, v in scenario_a.items():
        row_a[k] = v
    row_b = base_row.copy()
    for k, v in scenario_b.items():
        row_b[k] = v

    out = pd.DataFrame([row_a, row_b]).reset_index(drop=True)
    out.insert(0, "scenario_label", [label_a, label_b])
    out[f"P({PRIMARY_TARGET})"] = model_primary.predict_proba(out[FEATURE_COLS])[:, 1]
    out[f"P({EXPANSION_TARGET})"] = model_expansion.predict_proba(out[FEATURE_COLS])[:, 1]
    return out

def interpret(out: pd.DataFrame) -> str:
    a = out.iloc[0]
    b = out.iloc[1]
    primary_pref = a["scenario_label"] if a[f"P({PRIMARY_TARGET})"] >= b[f"P({PRIMARY_TARGET})"] else b["scenario_label"]
    expansion_pref = a["scenario_label"] if a[f"P({EXPANSION_TARGET})"] >= b[f"P({EXPANSION_TARGET})"] else b["scenario_label"]
    verdict = "AGREE" if primary_pref == expansion_pref else "DISAGREE"
    return f"{verdict}: {PRIMARY_TARGET} prefers {primary_pref!r}; {EXPANSION_TARGET} prefers {expansion_pref!r}."

def pair_ver_bahrain():
    return dict(
        context_filter={"season": 2023, "circuit": "Bahrain Grand Prix", "Driver": "VER"},
        scenario_a={
            "n_stops": 1, "strategy_type": "one_stop",
            "compound_sequence": "S-M",
            "stint1_length": 40, "stint2_length": 0, "stint3_length": 0,
            "avg_pit_stop_duration_s": 22, "total_pit_time_s": 22,
            "first_pit_lap": 20, "last_pit_lap": 20,
        },
        scenario_b={
            "n_stops": 2, "strategy_type": "two_stop",
            "compound_sequence": "S-M-S",
            "stint1_length": 22, "stint2_length": 20, "stint3_length": 0,
            "avg_pit_stop_duration_s": 22, "total_pit_time_s": 44,
            "first_pit_lap": 18, "last_pit_lap": 40,
        },
        label_a="1-stop S-M",
        label_b="2-stop S-M-S",
    )

pair = pair_ver_bahrain()
out = score_pair(test, **pair)
print(out[["scenario_label", "season", "circuit", "Driver", "Team", "grid_position",
           "n_stops", "compound_sequence", "stint1_length", "stint2_length", "stint3_length",
           f"P({PRIMARY_TARGET})", f"P({EXPANSION_TARGET})"]].to_string(index=False))
print()
print(interpret(out))


## Step 5 — Side-by-side visualization

This chart is the central Hito 2 artifact for the what-if comparison.


In [ ]:
# ===== Step 5 — Side-by-side plot =====
fig, ax = plt.subplots(figsize=(8, 4.5))
labels = list(out["scenario_label"])
x = np.arange(len(labels))
width = 0.35

primary_vals = out[f"P({PRIMARY_TARGET})"].values
ax.bar(x - width/2, primary_vals, width, label=f"P({PRIMARY_TARGET})", color="#1A3A5C")
for xi, v in zip(x - width/2, primary_vals):
    ax.text(xi, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

exp_vals = out[f"P({EXPANSION_TARGET})"].values
ax.bar(x + width/2, exp_vals, width, label=f"P({EXPANSION_TARGET})", color="#E63946")
for xi, v in zip(x + width/2, exp_vals):
    ax.text(xi, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Probability")
ax.set_title("Matched what-if pair · context held fixed · strategy inputs varied")
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Step 6 — Error analysis

The rubric asks for slices by strategy type, circuit type, and one additional context. This notebook uses **constructor tier** as the extra context because it is pre-race, operationally meaningful, and it creates a clear separation in class balance.


In [ ]:
# ===== Step 6 — Error analysis tables =====
def strategy_bucket(n_stops):
    if pd.isna(n_stops):
        return "unknown"
    n_stops = int(n_stops)
    if n_stops <= 0:
        return "no_stop"
    if n_stops == 1:
        return "one_stop"
    if n_stops == 2:
        return "two_stop"
    return "three_plus_stop"

analysis_df = test.copy()
analysis_df["strategy_bucket"] = analysis_df["n_stops"].apply(strategy_bucket)
analysis_df[f"P({PRIMARY_TARGET})"] = model_primary.predict_proba(analysis_df[FEATURE_COLS])[:, 1]
analysis_df[f"P({EXPANSION_TARGET})"] = model_expansion.predict_proba(analysis_df[FEATURE_COLS])[:, 1]

def slice_metrics(df_slice: pd.DataFrame, target: str, proba_col: str) -> dict:
    y = df_slice[target]
    p = df_slice[proba_col]
    return {
        "n": int(len(df_slice)),
        "base_rate": float(y.mean()),
        "brier": float(brier_score_loss(y, p)),
        "log_loss": float(log_loss(y, p, labels=[0, 1])),
        "roc_auc": float(roc_auc_score(y, p)) if y.nunique() > 1 else np.nan,
    }

def build_slice_table(group_col: str) -> pd.DataFrame:
    rows = []
    for group_value, df_slice in analysis_df.groupby(group_col, dropna=False):
        for target in [PRIMARY_TARGET, EXPANSION_TARGET]:
            proba_col = f"P({target})"
            metrics = slice_metrics(df_slice, target, proba_col)
            metrics["slice"] = group_value
            metrics["target"] = target
            rows.append(metrics)
    table = pd.DataFrame(rows)
    return table[["slice", "target", "n", "base_rate", "brier", "log_loss", "roc_auc"]].sort_values(["slice", "target"]).reset_index(drop=True)

strategy_table = build_slice_table("strategy_bucket")
circuit_table = build_slice_table("circuit_type")
constructor_table = build_slice_table("constructor_tier")

print("Strategy slices")
display(strategy_table)
print("
Circuit type slices")
display(circuit_table)
print("
Constructor tier slices")
display(constructor_table)

comparison = pd.DataFrame({
    "scenario": out["scenario_label"],
    PRIMARY_TARGET: out[f"P({PRIMARY_TARGET})"],
    EXPANSION_TARGET: out[f"P({EXPANSION_TARGET})"],
})
print("
Matched what-if summary")
display(comparison)


## Step 7 — Submission checklist

- `is_top10` and `is_top3` are both modeled with the same locked temporal split.
- The what-if pair shows a target disagreement, which is the key Hito 2 finding.
- The markdown deliverables in `capstone/hito_2` summarize the comparison, slices, leakage guard, mitigations, and AI-assisted reasoning.
- Before submission, re-run all cells from top to bottom to confirm the notebook is reproducible.
